# Kata 05 — Extracción Estructurada Defensiva

> Spec: [`specs/005-defensive-extraction/spec.md`](../../specs/005-defensive-extraction/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

# Kata más exigente con extracción → uso Sonnet por defecto
client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Para extraer datos de texto crudo, fuerzo al modelo a usar un *tool call* con JSON Schema estricto: required reales, opcionales nullable, enums con válvula de escape (`other`, `unclear`) acompañadas de un campo `_details`. La validación falla cerrada si el modelo intenta inventar.

## 2. Por qué importa

Pedir "devuelve JSON" en prosa garantiza fabricación: el modelo prefiere inventar a admitir incertidumbre. El schema lo obliga a admitirla.

## 3. Ejemplo correcto — schema con escape para ambigüedad

In [ ]:
import json

EXTRACT_TICKET = {
    "name": "extract_ticket",
    "description": "Estructura un ticket de soporte.",
    "input_schema": {
        "type": "object",
        "properties": {
            "summary": {"type": "string"},
            "category": {
                "type": "string",
                "enum": ["billing", "auth", "performance", "feature_request", "other"],
            },
            "category_other_details": {"type": ["string", "null"]},
            "urgency": {
                "type": "string",
                "enum": ["low", "medium", "high", "unclear"],
            },
            "customer_id": {"type": ["string", "null"]},
        },
        "required": ["summary", "category", "urgency"],
        "additionalProperties": False,
    },
}

def extract(client, text: str) -> dict:
    system = (
        "Estructura el ticket. Si el campo no aparece en el texto, usa null o el enum de escape "
        "('other' con detalles, 'unclear' para urgencia). NUNCA inventes datos."
    )
    resp = client.messages.create(
        system=system,
        messages=[{"role": "user", "content": text}],
        tools=[EXTRACT_TICKET],
        tool_choice={"type": "tool", "name": "extract_ticket"},
    )
    tu = next(b for b in resp.content if b.type == "tool_use")
    return tu.input

In [ ]:
cases = [
    "No me llega la factura desde hace 3 meses. Cliente C-123.",
    "Tengo una idea cool para mejorar el onboarding.",
    "Algo no funciona, ayuda.",
    "El sistema responde pero raro, como si fuera más lento, no estoy seguro.",
]
for t in cases:
    print(json.dumps(extract(client, t), indent=2, ensure_ascii=False))
    print("---")

Observa los casos 3 y 4: el modelo debe admitir incertidumbre con `unclear`/`other` o nulls — no fabrica un `customer_id` ni una urgencia inventada.

## 4. Anti-patrón — pedir JSON en prosa, sin schema

In [ ]:
def extract_prose(client, text: str) -> str:
    system = "Devuelve un JSON con summary, category, urgency, customer_id."
    resp = client.messages.create(
        system=system,
        messages=[{"role": "user", "content": text}],
    )
    return next((b.text for b in resp.content if b.type == "text"), "")

print(extract_prose(client, cases[3]))

Sin schema el modelo:

- inventa `customer_id` aunque no aparezca,
- elige una `urgency` cualquiera ("medium" suele ser el default fabricado),
- puede devolver el JSON envuelto en markdown o con campos extras.

Ningún parser determinista puede confiar en esa salida.

## 5. Argumento de certificación

- **`tool_choice` forzado** evita la salida en prosa.
- **`additionalProperties: false`** impide que el modelo añada campos imaginados.
- **Enums con `other` + `_details`**: dan al modelo una vía honesta para reconocer ambigüedad sin romper el contrato.
- **`required` mínimo**: sólo lo que **realmente** aparece siempre. Lo demás es nullable union.
- **Conexión con otros katas**: el schema es la misma técnica de Kata 13 (review CI), Kata 16 (handoff), Kata 20 (claims con provenance).

## 6. Auto-evaluación

**1. Si el dato no aparece en la fuente, ¿qué valor debe poner el modelo?**

`null` para campos nullable; el enum de escape (`other`, `unclear`) para enums. **Nunca** un valor inventado plausible.

**2. ¿Por qué `category_other_details` es necesario al lado del enum?**

Si elige `other`, necesito saber qué quiso decir. Sin el campo paralelo el `other` no aporta información — es básicamente un null. Con él, el agente registra la categoría no contemplada y yo decido si vale la pena agregarla al enum.

**3. ¿Qué prueba demuestra que el schema bloquea fabricación cuando reintroduzco `required` excesivos?**

Si marco `customer_id` como required, en los casos 2 y 4 el modelo se ve forzado a inventarlo. Comparar la salida del extractor con `customer_id: required` vs nullable demuestra el impacto. La aserción defensiva: pasar un texto sin id explícito y verificar que el extractor con schema correcto devuelve `null`.